In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("Dataset/dirty_cafe_sales.csv")
df_raw = df.copy()

print("Shape (Rows, Columns):", df.shape)
print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Info ---")
print(df.info())

quality_report = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "unique_count": df.nunique(dropna=True),
    "sample_values": [df[col].dropna().unique()[:5] for col in df.columns],
})

print("\n--- Data Quality Report ---")
display(quality_report)

print("\nExact duplicate rows:", df.duplicated().sum())

Shape (Rows, Columns): (10000, 8)

--- First 5 Rows ---


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11



--- Info ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB
None

--- Data Quality Report ---


,dtype,null_count,null_pct,unique_count,sample_values
Transaction ID,str,0,0.00,10000,"[TXN_1961373, TXN_4977031, TXN_4271903, TXN_70..."
Item,str,333,3.33,10,"[Coffee, Cake, Cookie, Salad, Smoothie]"
Quantity,str,138,1.38,7,"[2, 4, 5, 3, 1]"
Price Per Unit,str,179,1.79,8,"[2.0, 3.0, 1.0, 5.0, 4.0]"
Total Spent,str,173,1.73,19,"[4.0, 12.0, ERROR, 10.0, 20.0]"
Payment Method,str,2579,25.79,5,"[Credit Card, Cash, UNKNOWN, Digital Wallet, E..."
Location,str,3265,32.65,4,"[Takeaway, In-store, UNKNOWN, ERROR]"
Transaction Date,str,159,1.59,367,"[2023-09-08, 2023-05-16, 2023-07-19, 2023-04-2..."



Exact duplicate rows: 0


### Data Quality Report Findings
After inspecting `dirty_cafe_sales.csv` (10,000 rows and 8 columns), here is what stands out before cleaning:

- **All columns loaded as `str`:** Even columns that should be numeric (`Quantity`, `Price Per Unit`, `Total Spent`) or dates (`Transaction Date`) are stored as text (`str`), likely because of invalid text values mixed in.
- **Missing values (`NaN`):** Every column except `Transaction ID` has missing values. `Location` has the highest missing rate (3,265 rows / 32.65%) and `Payment Method` has 2,579 missing rows (25.79%), while `Item` (333), `Price Per Unit` (179), `Total Spent` (173), `Transaction Date` (159), and `Quantity` (138) have smaller shares.
- **Placeholder/error text:** Values like `"ERROR"` and `"UNKNOWN"` appear inside columns such as `Total Spent`, `Payment Method`, and `Location`, so they need to be treated as missing/invalid rather than real categories or numbers.
- **Exact duplicates:** There are `0` exact full-row duplicates at first glance, though we should still check carefully once strings and placeholders are standardized.
- **Business logic & range check:** In cafe sales data, `Quantity`, `Price Per Unit`, and `Total Spent` should be positive values and follow `Total Spent = Quantity × Price Per Unit` where possible, so we will check for mismatches or odd values as we clean.

In [2]:
# 1. تنظيف المسافات وتوحيد القيم الفاضية أو النصية الغلط (ERROR / UNKNOWN) إلى NaN
placeholders = ["", " ", "NA", "N/A", "nan", "NaN", "NONE", "None", "ERROR", "error", "UNKNOWN", "unknown"]

for col in df.columns:
    # نتعامل مع القيم النصية ونشيل المسافات من الأول والآخر
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    df[col] = df[col].replace(placeholders, np.nan)

# 2. تحويل الأعمدة الرقمية إلى float
num_cols = ["Quantity", "Price Per Unit", "Total Spent"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Nulls after converting placeholders & numeric types:")
print(df.isnull().sum())
print("\nCurrent dtypes:")
print(df.dtypes)

Nulls after converting placeholders & numeric types:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

Current dtypes:
Transaction ID          str
Item                    str
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method          str
Location                str
Transaction Date        str
dtype: object


### Placeholder Cleanup & Initial Numeric Conversion
Replacing text placeholders (like `"ERROR"` and `"UNKNOWN"`), trimming whitespace, and converting `Quantity`, `Price Per Unit`, and `Total Spent` to `float64` revealed the true missing counts:
- **Higher missingness after placeholder cleanup:** Columns like `Item` (969), `Payment Method` (3,178), and `Location` (3,961) jumped up once `"ERROR"` and `"UNKNOWN"` were treated as missing instead of valid categories.
- **Numeric fields ready for math:** `Quantity` (479 nulls), `Price Per Unit` (533 nulls), and `Total Spent` (502 nulls) are now `float64`, which lets us check relationships between them (`Total Spent = Quantity × Price Per Unit`) before deciding how to fill remaining gaps.

In [3]:
# نبدأ من نسخة نظيفة من الداتا الأصلية عشان نلتزم بالترتيب خطوة بخطوة
df = df_raw.copy()

# 1. توحيد القيم النصية الفارغة / ERROR / UNKNOWN إلى NaN
placeholders = [
    "",
    " ",
    "NA",
    "N/A",
    "nan",
    "NaN",
    "NONE",
    "None",
    "ERROR",
    "error",
    "UNKNOWN",
    "unknown",
]
for col in df.columns:
  df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
  df[col] = df[col].replace(placeholders, np.nan)

# 2. تحويل الأعمدة الرقمية مؤقتاً لأرقام عشان نحسب median صح
num_cols = ["Quantity", "Price Per Unit", "Total Spent"]
for col in num_cols:
  df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. تطبيق استراتيجيات معالجة القيم الناقصة لكل عمود
# - Median imputation للأعمدة الرقمية
for col in num_cols:
  df[col] = df[col].fillna(df[col].median())

# - Mode imputation للأعمدة النصية/الفئوية
cat_cols = ["Item", "Payment Method", "Location"]
for col in cat_cols:
  df[col] = df[col].fillna(df[col].mode()[0])

# - Forward fill لعمود التاريخ
df["Transaction Date"] = df["Transaction Date"].ffill()

print("Nulls after Missing Data Handling:")
print(df.isnull().sum())

Nulls after Missing Data Handling:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


### Missing Data Handling
After converting text placeholders (such as `"ERROR"` and `"UNKNOWN"`) to `NaN`, I handled missing values column by column using clear strategies:
- **Numeric columns (`Quantity`, `Price Per Unit`, `Total Spent`):** Filled missing values using the **median** so extreme values or mixed price/quantity rows do not skew the replacement.
- **Categorical columns (`Item`, `Payment Method`, `Location`):** Filled missing values using the **mode** (most frequent valid value) so categories stay consistent with the dataset.
- **Date column (`Transaction Date`):** Used **forward fill** to carry forward the previous valid date without dropping rows.

In [4]:
# عدد الصفوف قبل حذف التكرارات
rows_before_dup = len(df)

# فحص التكرارات الكاملة (أو تكرار نفس العملية في كل الأعمدة ما عدا Transaction ID لو حابة تتحققي)
full_duplicates = df.duplicated().sum()
print(f"Exact full-row duplicates found: {full_duplicates}")

# حذف التكرارات الكاملة إن وُجدت
df = df.drop_duplicates().reset_index(drop=True)

rows_after_dup = len(df)
print(f"Rows removed: {rows_before_dup - rows_after_dup}")
print(f"Total rows remaining: {rows_after_dup}")

Exact full-row duplicates found: 0
Rows removed: 0
Total rows remaining: 10000


### Duplicate Removal
Checked the dataset for duplicate rows after cleaning placeholders and filling missing values:
- **Exact full-row duplicates:** Found `0` duplicate rows, so `0` rows were removed.
- **Row count:** All `10,000` rows remain intact with the index reset cleanly.

In [5]:
# 1. توحيد تنسيق الأعمدة النصية (إزالة المسافات الزيادة وتوحيد الحروف Title Case)
text_cols = ["Item", "Payment Method", "Location"]
for col in text_cols:
  df[col] = df[col].astype(str).str.strip().str.title()

# نظرة سريعة على القيم الفريدة بعد التوحيد للتأكد إن مفيش تكرار بصيغ مختلفة
for col in text_cols:
  print(f"Unique values in {col}: {df[col].unique().tolist()}")

# 2. تحويل Transaction Date إلى datetime وتوحيد التنسيق
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

# لو ظهر أي NaT بعد التحويل، نكمله بـ forward fill و backward fill كاحتياط
df["Transaction Date"] = df["Transaction Date"].ffill().bfill()

print("\nTransaction Date dtype:", df["Transaction Date"].dtype)
print("Any NaT in Transaction Date?:", df["Transaction Date"].isna().sum())

Unique values in Item: ['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'Juice', 'Sandwich', 'Tea']
Unique values in Payment Method: ['Credit Card', 'Cash', 'Digital Wallet']
Unique values in Location: ['Takeaway', 'In-Store']

Transaction Date dtype: datetime64[us]
Any NaT in Transaction Date?: 0


### Standardisation
Standardised text formatting and date values across the dataset so categories and dates follow a single consistent format:
- **Categorical text formatting:** Stripped extra whitespace and applied consistent title casing across `Item`, `Payment Method`, and `Location` so variations in casing or spacing do not split the same category.
- **Date standardisation:** Parsed `Transaction Date` into proper `datetime` format (`datetime64[ns]`) and ensured no invalid or `NaT` dates remained.

In [6]:
num_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in num_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR

  outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
  print(f"--- {col} ---")
  print(
      f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR} | Bounds: [{lower_bound}, {upper_bound}]"
  )
  print(f"Outliers detected: {outliers_count}")

  # تطبيق Capping (حصر القيم المتطرفة داخل الحدود المنطقية لـ IQR)
  df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("\nOutliers check after capping:")
for col in num_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  rem = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
  print(f"{col}: {rem} outliers remaining")

--- Quantity ---
Q1: 2.0, Q3: 4.0, IQR: 2.0 | Bounds: [-1.0, 7.0]
Outliers detected: 0
--- Price Per Unit ---
Q1: 2.0, Q3: 4.0, IQR: 2.0 | Bounds: [-1.0, 7.0]
Outliers detected: 0
--- Total Spent ---
Q1: 4.0, Q3: 12.0, IQR: 8.0 | Bounds: [-8.0, 24.0]
Outliers detected: 259

Outliers check after capping:
Quantity: 0 outliers remaining
Price Per Unit: 0 outliers remaining
Total Spent: 0 outliers remaining


### Outlier Detection & Handling
Checked numeric columns (`Quantity`, `Price Per Unit`, and `Total Spent`) for outliers using the **IQR** method (`Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR`):
- **Findings:** `Quantity` and `Price Per Unit` had `0` outliers, while `Total Spent` had `259` values above the upper bound (`24.0`).
- **Decision (Capping):** Applied **capping** (`clip`) on `Total Spent` so all `259` extreme values were bounded to `24.0` while keeping all rows intact, leaving `0` outliers across all three numeric columns.

In [7]:
# ضبط الأنواع النهائية لكل عمود بوضوح
string_cols = ["Transaction ID", "Item", "Payment Method", "Location"]
for col in string_cols:
  df[col] = df[col].astype("string")

float_cols = ["Quantity", "Price Per Unit", "Total Spent"]
for col in float_cols:
  df[col] = df[col].astype("float64")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

print("--- Final Data Types ---")
print(df.dtypes)
print("\n--- Any Remaining Nulls ---")
print(df.isnull().sum())

--- Final Data Types ---
Transaction ID              string
Item                        string
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              string
Location                    string
Transaction Date    datetime64[us]
dtype: object

--- Any Remaining Nulls ---
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


### Data Type Correction
Verified and enforced the final data types across all columns so each field matches its role:
- **IDs & text categories (`string`):** Ensured `Transaction ID`, `Item`, `Payment Method`, and `Location` are stored as `string`.
- **Numeric & monetary values (`float64`):** Confirmed `Quantity`, `Price Per Unit`, and `Total Spent` are `float64`.
- **Date (`datetime`):** Kept `Transaction Date` parsed as a datetime type (`datetime64[us]`) with zero `NaT` values.

In [10]:
# حساب المؤشرات قبل التنظيف (من df_raw)
raw_rows = len(df_raw)
raw_nulls = int(df_raw.isnull().sum().sum())
raw_dups = int(df_raw.duplicated().sum())
# في الداتا الخام، كل الأعمدة كانت str بما فيها الأرقام والتواريخ (8 أعمدة وكلها str)
raw_correct_dtypes = "0 / 8 (all loaded as str)"

# حساب المؤشرات بعد التنظيف (من df)
clean_rows = len(df)
clean_nulls = int(df.isnull().sum().sum())
clean_dups = int(df.duplicated().sum())

expected_dtypes = {
    "Transaction ID": "string",
    "Item": "string",
    "Quantity": "float64",
    "Price Per Unit": "float64",
    "Total Spent": "float64",
    "Payment Method": "string",
    "Location": "string",
}
matched = sum(
    str(df[col].dtype) == exp for col, exp in expected_dtypes.items()
) + (1 if "datetime" in str(df["Transaction Date"].dtype) else 0)
clean_correct_dtypes = f"{matched} / {len(df.columns)} (all matched)"

summary_df = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Missing Values (Nulls)",
        "Exact Duplicate Rows",
        "Data Type Accuracy",
    ],
    "Before Cleaning": [raw_rows, raw_nulls, raw_dups, raw_correct_dtypes],
    "After Cleaning": [clean_rows, clean_nulls, clean_dups, clean_correct_dtypes],
})

display(summary_df)

,Metric,Before Cleaning,After Cleaning
0,Total Rows,10000,10000
1,Total Missing Values (Nulls),6826,0
2,Exact Duplicate Rows,0,0
3,Data Type Accuracy,0 / 8 (all loaded as str),8 / 8 (all matched)


### Before vs. After Summary
The summary table compares the dataset before and after cleaning:
- **Total rows:** Remained `10,000` rows with `0` exact duplicate rows removed.
- **Missing values:** Reduced total missing values across the dataset from `6,826` to `0`.
- **Data types:** Improved from `0 / 8` (all loaded as `str`) to `8 / 8` matching their expected types (`string`, `float64`, and `datetime`).

In [11]:
output_path = "Dataset/cleaned_cafe_sales.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved successfully to: {output_path}")

Cleaned dataset saved successfully to: Dataset/cleaned_cafe_sales.csv


### Saving the Cleaned Dataset
Exported the final cleaned dataset to `Dataset/cleaned_cafe_sales.csv` (`index=False`) so it is ready for downstream analysis or visualization without repeating the cleaning steps.